# 음원 분리 실제 곡 품질 평가 — GPU 서버 실행 (S15P31A307-166)

음원 분리 모델 BS Roformer SW를 실제 드러머 녹음에 돌려 분리 품질을 잰다. 대표 측정은 ENST-Drums 반주 있는 64곡이다. ENST는 드럼(wet_mix)과 반주를 우리가 직접 더해 원곡을 만들기 때문에 정답 드럼이 원곡 속 드럼과 정확히 같아서 드럼 SDR을 믿을 수 있다. MDB Drums는 정답 드럼 트랙이 원곡 속 드럼과 달라 SDR을 쓸 수 없고, 10/29에 돌린 MDB run 두 개는 재현용으로 6번 셀에 남겨 둔다. **위에서부터 차례로 실행한다.** 셀마다 명령만 부르고 로직은 없다.

**준비물.** 로컬 `AI/`에서 `uv run python scripts/pack_server_upload.py --code-only`로 만든 `data/upload/server/S15P31A307_code.zip`을 JupyterHub `ssaplash/upload/`에 올린다(파일 목록 화면 오른쪽 위 Upload). 이 노트북은 `ssaplash/notebooks/`에 올린다. 서버에 이미 있는 것을 그대로 쓴다: 코드 폴더의 `.venv`(분리 모델 패키지 포함), `data/mdb_drums`, `data/models/bs_roformer_sw`(10/29 받음), ENST 원본 `ssaplash/datasets/enst-drums/`, ffmpeg `ssaplash/tools/ffmpeg-7.0.2-amd64-static/`.

**순서.** 1 설정 → 2 코드 덮어 풀기(1분) → 3 ENST 링크·데이터 점검(1~2분) → 4 30초 시험(2~3분) → 5 ENST 본측정(10분 안팎) → 6 MDB 재현(필요할 때만) → 7 결과 묶기.

**끊기면.** 커널이 다시 시작됐으면 1번 셀을 다시 실행하고 끊긴 셀을 그대로 다시 실행한다. run-id가 고정돼 있어 끝난 곡은 건너뛰고 실패했거나 남은 곡만 돈다.

In [ ]:
# 1. 설정. 지정받은 GPU는 6번. ffmpeg 폴더를 PATH 앞에 붙임 (점수 계산 패키지 museval이 불러오는 stempeg이 ffmpeg·ffprobe를 찾음)
import os
import sys

GPU_ID = "6"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONUTF8"] = "1"
os.environ["MPLBACKEND"] = "Agg"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"
PY = sys.executable
UPLOAD = f"{WORK}/upload"
RESULTS = f"{WORK}/results"
FFMPEG = f"{WORK}/tools/ffmpeg-7.0.2-amd64-static"
os.environ["PATH"] = f"{FFMPEG}:{os.environ['PATH']}"
os.makedirs(RESULTS, exist_ok=True)
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!df -h $HOME | tail -1
!ls -la $UPLOAD/S15P31A307_code.zip
!which ffmpeg ffprobe
# 확인: GPU 한 줄(H200, 남은 메모리 10GB 이상이면 충분), 디스크 여유 10GB 이상, 코드 zip이 오늘 날짜,
#       ffmpeg·ffprobe 두 줄이 ssaplash/tools/ffmpeg-7.0.2-amd64-static/ 아래로 찍힘

In [ ]:
# 2. 코드 덮어 풀기. 기존 폴더를 지우지 않고 위에 덮어씀 (.venv·데이터 링크·data/models·지난 결과가 그대로 남음)
!$PY -m zipfile -e $UPLOAD/S15P31A307_code.zip $WORK
%cd $AI
!ls src/ssaplash_ai/bench/datasets/enst.py configs/bench/datasets/enst_minus_one.yaml configs/bench/models/msst/bs_roformer_sw.yaml
!ls data/models/bs_roformer_sw
!.venv/bin/python -c "import torch, msst; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))"
# 확인: 파일 3개가 그대로 찍힘. data/models/bs_roformer_sw 에 BS-Rofo-SW-Fixed.ckpt·BS-Rofo-SW-Fixed.yaml.
#       마지막 줄 'torch 2.8.0+cu128 cuda True NVIDIA H200 NVL'. enst.py가 없다고 나오면 zip을 새로 만들었는지 확인

In [ ]:
# 3. ENST 링크 + 데이터 점검 (여러 번 쳐도 됨). 링크는 이미 있으면 같은 곳으로 다시 걸림
!mkdir -p data/enst && ln -sfn $WORK/datasets/enst-drums/ENST-drums-public data/enst/ENST-drums-public
!ls data/enst/ENST-drums-public
!.venv/bin/ssaplash-bench check-data configs/bench/datasets/enst_minus_one.yaml configs/bench/datasets/mdb_drums.yaml configs/bench/datasets/mdb_drums_unseen.yaml
# 확인: ls에 drummer_1 drummer_2 drummer_3. check-data가
#       [enst_minus_one] 곡 64개 (기대 64), 길이 합 60.9분, 누락 0건 / 상태: 정상 (타격 정답 26140개)
#       [mdb_drums] 곡 23개 (기대 23), 길이 합 21.8분, 누락 0건
#       [mdb_drums_unseen] 곡 11개 (기대 11), 길이 합 13.6분, 누락 0건
#       누락이 있으면 그 줄을 적어 두고 멈춤

In [ ]:
# 4. 30초 시험: ENST 64곡의 앞 30초. 끝에 '실패 0개'인지 확인 (여러 번 쳐도 됨)
M = "configs/bench/models/msst/bs_roformer_sw.yaml"
DS = "configs/bench/datasets"
RUN = f".venv/bin/ssaplash-bench run --models {M} --out data/results --mlflow-uri none"
QUIET = "2>&1 | grep -v 'Glyph\\|fig\\.\\(tight_layout\\|savefig\\)'"  # 서버에 한글 글꼴이 없어 나는 그래프 경고만 거름
!$RUN --dataset $DS/enst_minus_one.yaml --clip 0 30 --run-id sep-quality-enst-clip30 $QUIET | tail -4
# 확인: 'run sep-quality-enst-clip30: 행 64개, 실패 0개 → data/results/sep-quality-enst-clip30/results.csv'
#       'ffmpeg or ffprobe could not be found'가 나오면 1번 셀을 다시 실행 (PATH가 커널에 안 들어간 것)

In [ ]:
# 5. ENST 본측정 64곡. 분리 wav는 저장하지 않음(64곡이면 약 2GB). 들어 볼 곡은 로컬에서 따로 분리함
!$RUN --dataset $DS/enst_minus_one.yaml --run-id sep-quality-enst $QUIET
# 확인: 곡마다 '[bs_roformer_sw] drummer_N/...: sdrDrums=... onsetF1=... rtf=...' 한 줄, 끝에
#       'run sep-quality-enst: 행 64개, 실패 0개 → data/results/sep-quality-enst/results.csv'.
#       실패가 남으면 이 셀을 한 번 더 실행 (실패한 곡만 다시 돔). 그래도 남으면 results.csv의 error 열을 적어 둠

In [ ]:
# 6. MDB 재현 (10/29에 이미 돌렸으면 건너뜀). 다시 실행해도 끝난 곡은 건너뛰어 금방 끝남
#    MDB는 정답 드럼 트랙이 원곡 속 드럼과 달라 드럼 SDR을 쓰지 않음. 타격 보존율만 봄
!$RUN --dataset $DS/mdb_drums_unseen.yaml --save-wav 11 --run-id sep-quality-mdbu $QUIET | tail -3
!$RUN --dataset $DS/mdb_drums.yaml --run-id sep-quality-mdb $QUIET | tail -3
# 확인: 'run sep-quality-mdbu: 행 11개, 실패 0개', 'run sep-quality-mdb: 행 23개, 실패 0개'

In [ ]:
# 7. 결과 묶기 (여러 번 쳐도 됨, 매번 새로 만듦). results 폴더의 sep_quality_enst.zip 을 내려받아 로컬 AI/data/results/ 에서 푼다
!cd data/results && $PY -m zipfile -c $RESULTS/sep_quality_enst.zip sep-quality-enst
!ls -la $RESULTS/sep_quality_enst.zip
!md5sum $RESULTS/sep_quality_enst.zip
# 확인: 1MB 안쪽 zip 하나(results.csv·summary.md·그래프). md5 값은 내려받은 파일과 비교할 때 씀